# IT Support Swarm — Local Test Notebook

Test the full agent pipeline interactively. Every step is printed so you can see exactly what each agent does.

**Run order:**
1. Setup & imports
2. Individual component tests (guardrail, CAG, memory)
3. Full swarm test with a real ticket
4. Try your own questions

## 1. Setup

In [2]:
import sys
import os
from pathlib import Path

# Add src/ to path so imports work from the notebook
PROJECT_ROOT = Path.cwd().parent
SRC_DIR      = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))
sys.path.insert(0, str(PROJECT_ROOT))

# Load .env
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

print("✓ Path configured")
print(f"  Project root : {PROJECT_ROOT}")
print(f"  Src dir      : {SRC_DIR}")

# Verify key env vars are loaded
for key in ["GROQ_API_KEY", "OPENROUTER_API_KEY", "SUPABASE_URL", "QDRANT_URL"]:
    val = os.getenv(key, "")
    status = "✓" if val else "✗ MISSING"
    print(f"  {status} {key}: {val[:20]}..." if val else f"  {status} {key}")

✓ Path configured
  Project root : c:\Users\viraj\Zuu\AEE\ResolutionAgent_FinalHackathon
  Src dir      : c:\Users\viraj\Zuu\AEE\ResolutionAgent_FinalHackathon\src
  ✓ GROQ_API_KEY: gsk_hHWKBHXhFEEogZvU...
  ✓ OPENROUTER_API_KEY: sk-or-v1-0bfb61bb3d5...
  ✓ SUPABASE_URL: https://ydkpfywkdxio...
  ✓ QDRANT_URL: https://54249fc0-384...


In [3]:
# Configure logging to show in notebook
from infastructure.log import configure_logging
configure_logging()
print("✓ Logging configured")

2026-07-05 14:20:56.820 | DEBUG    | infastructure.config:get_config:50 - Loaded param.yaml (environment=development)


2026-07-05T14:20:56.820537+0530 | INFO | infastructure.log | Logging configured | level=INFO env=development
✓ Logging configured


## 2. Connection Health Checks

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

print("SUPABASE_URL              :", os.getenv("SUPABASE_URL"))
print("SUPABASE_KEY :", os.getenv("SUPABASE_KEY", "")[:20] + "...")
print("AZURE_STORAGE_CONNECTION_STRING :", os.getenv("AZURE_STORAGE_CONNECTION_STRING", "")[:30] + "...")

SUPABASE_URL              : https://ydkpfywkdxionxfukpov.supabase.co
SUPABASE_KEY : sb_secret_LJ44mlZzNo...
AZURE_STORAGE_CONNECTION_STRING : DefaultEndpointsProtocol=https...


In [5]:
import asyncio
from infastructure.db.supabase_client import check_connection as supabase_health
from infastructure.db.qdrant_client   import check_connection as qdrant_health
from services.azure.blob_service_client import check_connection as azure_health

print("Checking connections...\n")

sb_ok     = await supabase_health()
qdrant_ok = await qdrant_health()
azure_ok  = await azure_health()

print(f"  Supabase  : {'✓ OK' if sb_ok     else '✗ FAILED'}")
print(f"  Qdrant    : {'✓ OK' if qdrant_ok else '✗ FAILED'}")
print(f"  Azure Blob: {'✓ OK' if azure_ok  else '✗ FAILED'}")

Checking connections...

  Supabase  : ✓ OK
  Qdrant    : ✓ OK
  Azure Blob: ✓ OK


## 3. Test Guardrail
First node in the graph. Should PASS IT tickets, FAIL off-topic or harmful inputs.

In [11]:
from agents.guardrail import guardrail_node
from agents.state import AgentState
import uuid

def make_state(ticket: str) -> AgentState:
    """Helper to build a minimal AgentState for testing."""
    return AgentState(
        ticket_id            = str(uuid.uuid4())[:8],
        ticket               = ticket,
        user_id              = "EMP-0150",
        is_safe              = False,
        guardrail_reason     = None,
        cag_hit              = False,
        cag_answer           = None,
        ticket_type          = None,
        severity             = None,
        classification_reason= None,
        findings             = None,
        similar_past_incidents=None,
        metrics_queried      = None,
        resolution           = None,
        runbook_used         = None,
        action_taken         = None,
        conversation_history = [],
        started_at           = None,
        resolved_at          = None,
        error                = None,
        retry_count          = 0,
    )

test_cases = [
    ("Redis is running out of memory on prod-cache-01", "should PASS"),
    ("My VPN keeps disconnecting",                     "should PASS"),
    ("What is the weather today?",                     "should FAIL"),
    ("lmao locked out again bro",                      "should PASS (slang)"),
]

print("=" * 60)
print("GUARDRAIL TESTS")
print("=" * 60)

for ticket, expected in test_cases:
    state  = make_state(ticket)
    result = await guardrail_node(state)
    verdict = "✓ PASS" if result["is_safe"] else "✗ FAIL"
    reason  = result.get("guardrail_reason") or ""
    print(f"\nTicket  : {ticket}")
    print(f"Expected: {expected}")
    print(f"Result  : {verdict}  {reason}")

2026-07-05T14:28:14.748821+0530 | INFO | infastructure.observability | Langfuse client initialised (host=https://cloud.langfuse.com)
2026-07-05T14:28:14.750076+0530 | WARNING | infastructure.observability | Failed to create Langfuse trace: 'Langfuse' object has no attribute 'trace'


GUARDRAIL TESTS

Ticket  : Redis is running out of memory on prod-cache-01
Expected: should PASS
Result  : ✓ PASS  

Ticket  : My VPN keeps disconnecting
Expected: should PASS
Result  : ✓ PASS  
2026-07-05T14:28:15.932697+0530 | INFO | agents.guardrail | Guardrail LLM blocked ticket e42f0da1: UNRELATED TO IT SUPPORT.

Ticket  : What is the weather today?
Expected: should FAIL
Result  : ✗ FAIL  UNRELATED TO IT SUPPORT.
2026-07-05T14:28:16.157649+0530 | INFO | agents.guardrail | Guardrail LLM blocked ticket 48d01f56: CONTAINS ABUSIVE LANGUAGE.

Ticket  : lmao locked out again bro
Expected: should PASS (slang)
Result  : ✗ FAIL  CONTAINS ABUSIVE LANGUAGE.


## 4. Test CAG Cache (Router)
Check if common questions get a fast-path cache hit.

In [12]:
from services.cag_service.cache import CAGCache

cache = CAGCache()

test_queries = [
    "How do I reset my VPN?",
    "I forgot my password",
    "Redis OOM on prod-cache-01",      # novel — should be a miss
    "My laptop won't turn on",
]

print("=" * 60)
print("CAG CACHE LOOKUP TESTS")
print("=" * 60)

for query in test_queries:
    answer = await cache.get(query)
    if answer:
        print(f"\n✓ HIT  : {query}")
        print(f"  Answer: {answer[:120]}...")
    else:
        print(f"\n✗ MISS : {query}")
        print(f"  → will route to swarm")

CAG CACHE LOOKUP TESTS
2026-07-05T14:29:48.428499+0530 | ERROR | services.cag_service.cache | CAGCache.get failed: 'AsyncQdrantClient' object has no attribute 'search'

✗ MISS : How do I reset my VPN?
  → will route to swarm
2026-07-05T14:29:49.368363+0530 | ERROR | services.cag_service.cache | CAGCache.get failed: 'AsyncQdrantClient' object has no attribute 'search'

✗ MISS : I forgot my password
  → will route to swarm
2026-07-05T14:29:50.278527+0530 | ERROR | services.cag_service.cache | CAGCache.get failed: 'AsyncQdrantClient' object has no attribute 'search'

✗ MISS : Redis OOM on prod-cache-01
  → will route to swarm
2026-07-05T14:29:50.705916+0530 | ERROR | services.cag_service.cache | CAGCache.get failed: 'AsyncQdrantClient' object has no attribute 'search'

✗ MISS : My laptop won't turn on
  → will route to swarm


## 5. Test L1 Triage
Classification agent — should output ticket_type (1-4) and severity.

In [ ]:
from agents.l1_triage import l1_triage_node

triage_tests = [
    "Redis is running out of memory on prod-cache-01",
    "I need a replacement laptop, mine has a cracked screen",
    "The checkout API is responding slowly, P99 is over 3 seconds",
    "lmao locked out again bro my mfa codes not working",
]

type_labels = {1: "Access & Identity", 2: "Asset Provisioning", 
               3: "Service Degradation", 4: "Critical Outage"}

print("=" * 60)
print("L1 TRIAGE TESTS")
print("=" * 60)

for ticket in triage_tests:
    state  = make_state(ticket)
    state["is_safe"] = True
    result = await l1_triage_node(state)
    
    t    = result.get("ticket_type", "?")
    sev  = result.get("severity", "?")
    why  = result.get("classification_reason", "")
    label = type_labels.get(t, "Unknown")
    
    print(f"\nTicket  : {ticket}")
    print(f"Type    : {t} — {label}")
    print(f"Severity: {sev}")
    print(f"Reason  : {why}")

## 6. Test Procedural Memory
Search Qdrant for similar past incidents (populated by distill pipeline).

In [ ]:
from memory.procedural_store import ProceduralStore

store = ProceduralStore()

# Check how many SOPs are in Qdrant
count = await store.count()
print(f"Procedural memory contains {count} SOPs\n")

# Search for similar past incidents
query       = "Redis running out of memory on production server"
ticket_type = 4

print(f"Searching for: '{query}'")
print(f"Ticket type  : {ticket_type} (Critical Outage)")
print("-" * 60)

results = await store.search_similar(query, ticket_type=ticket_type, top_k=3)

if not results:
    print("No results — run distill_stage1.py + distill_stage2.py first")
else:
    for i, r in enumerate(results, 1):
        print(f"\n[{i}] Score: {r['similarity_score']}")
        print(f"    Problem    : {r['problem']}")
        print(f"    Root Cause : {r['root_cause']}")
        print(f"    Resolution : {r['resolution']}")
        print(f"    Escalated  : {r['was_escalated']} | Satisfaction: {r['satisfaction_score']}/5")

## 7. Test RAG — Runbook Retrieval
Search Qdrant for relevant runbook chunks (populated by ingest_runbooks.py).

In [ ]:
from services.rag_service.chain import RAGChain

rag = RAGChain()

# List available runbooks
runbooks = await rag.get_runbook_list()
print(f"Available runbooks: {runbooks}\n")

# Test retrieval
query = "Redis memory full, OOM killer triggered on prod-cache-01"

print(f"Query: {query}")
print("-" * 60)

context, source = await rag.retrieve_and_format(query=query, ticket_type=4)

print(f"Runbook source : {source}")
print(f"\nContext injected into L3:")
print(context[:800])

## 8. Test Observability MCP
Query mock server metrics and service status.

In [ ]:
from mcp_servers.observability_server import ObservabilityMCPClient

obs = ObservabilityMCPClient()

print("=" * 60)
print("OBSERVABILITY MCP — Server Metrics")
print("=" * 60)

# Query specific server
metrics = await obs.get_metrics(
    ticket      = "Redis OOM on prod-cache-01",
    ticket_type = 4,
)

print("\nServer metrics:")
if metrics.get("server_metrics"):
    m = metrics["server_metrics"]
    print(f"  Server : {m.get('server_id')}")
    print(f"  CPU    : {m.get('cpu_percent')}%")
    print(f"  Memory : {m.get('memory_percent')}%")
    print(f"  Disk   : {m.get('disk_percent')}%")
    print(f"  Status : {m.get('status')}")
else:
    print("  No server metrics found")

print("\nUnhealthy services:")
for svc in metrics.get("unhealthy_services", []):
    print(f"  {svc.get('service_name'):20s} | {svc.get('status'):10s} | error_rate: {svc.get('error_rate_percent')}%")

## 9. Full Swarm Test — Watch Every Step
Run a ticket through the complete pipeline and print every agent's output.

In [ ]:
from agents.decision_graph import process_ticket

async def run_and_display(ticket: str, user_id: str = "EMP-0150"):
    """Run a ticket through the swarm and print every step clearly."""
    
    print("\n" + "=" * 70)
    print(f"TICKET: {ticket}")
    print("=" * 70)
    
    final_state = await process_ticket(ticket=ticket, user_id=user_id)
    
    # ── Guardrail ──────────────────────────────────────────────────────
    print("\n── GUARDRAIL ─────────────────────────────────────")
    if final_state.get("is_safe"):
        print("  ✓ PASSED")
    else:
        print(f"  ✗ BLOCKED: {final_state.get('guardrail_reason')}")
        return final_state
    
    # ── CAG Router ────────────────────────────────────────────────────
    print("\n── CAG ROUTER ────────────────────────────────────")
    if final_state.get("cag_hit"):
        print("  ✓ CACHE HIT — skipped swarm")
        print(f"  Answer: {final_state.get('cag_answer', '')[:120]}")
        return final_state
    else:
        print("  ✗ CACHE MISS — routing to swarm")
    
    # ── L1 Triage ─────────────────────────────────────────────────────
    type_labels = {1: "Access & Identity", 2: "Asset Provisioning",
                   3: "Service Degradation", 4: "Critical Outage"}
    t = final_state.get("ticket_type")
    print("\n── L1 TRIAGE ─────────────────────────────────────")
    print(f"  Type    : {t} — {type_labels.get(t, 'Unknown')}")
    print(f"  Severity: {final_state.get('severity')}")
    print(f"  Reason  : {final_state.get('classification_reason', '')}")
    
    # ── L2 Investigator ───────────────────────────────────────────────
    print("\n── L2 INVESTIGATOR ───────────────────────────────")
    findings = final_state.get("findings", "No findings")
    print(f"  {findings}")
    
    metrics = final_state.get("metrics_queried", {})
    if metrics.get("server_metrics"):
        m = metrics["server_metrics"]
        print(f"\n  Live metrics — {m.get('server_id')}: CPU {m.get('cpu_percent')}% | Mem {m.get('memory_percent')}%")
    
    past = final_state.get("similar_past_incidents", [])
    if past:
        print(f"\n  Past incidents found: {len(past)}")
        for p in past[:2]:
            print(f"    - {p.get('problem', '')} → {p.get('resolution', '')[:80]}")
    
    # ── L3 Resolver ───────────────────────────────────────────────────
    print("\n── L3 RESOLVER ───────────────────────────────────")
    print(f"  Runbook  : {final_state.get('runbook_used') or 'None'}")
    print(f"  Action   : {final_state.get('action_taken') or 'None'}")
    
    # ── Final Resolution ──────────────────────────────────────────────
    print("\n── RESOLUTION ────────────────────────────────────")
    print(final_state.get("resolution", "No resolution generated"))
    
    # ── Timing ────────────────────────────────────────────────────────
    if final_state.get("started_at") and final_state.get("resolved_at"):
        from datetime import datetime
        start = datetime.fromisoformat(final_state["started_at"])
        end   = datetime.fromisoformat(final_state["resolved_at"])
        secs  = (end - start).total_seconds()
        print(f"\n── TIMING ────────────────────────────────────────")
        print(f"  Total time: {secs:.1f}s")
    
    if final_state.get("error"):
        print(f"\n⚠ Error: {final_state['error']}")
    
    print("\n" + "=" * 70)
    return final_state

In [ ]:
# ── Test 1: Critical Outage (Type 4) ──────────────────────────────────
state = await run_and_display(
    "Redis is running out of memory on prod-cache-01, OOM killer triggered"
)

In [ ]:
# ── Test 2: Access & Identity (Type 1) — should hit CAG cache ─────────
state = await run_and_display(
    "How do I reset my VPN credentials?"
)

In [ ]:
# ── Test 3: Service Degradation (Type 3) ──────────────────────────────
state = await run_and_display(
    "The checkout API is super slow today, P99 latency is above 3 seconds"
)

In [ ]:
# ── Test 4: Guardrail block ────────────────────────────────────────────
state = await run_and_display(
    "What is the recipe for chocolate cake?"
)

## 10. Try Your Own Ticket
Change the message below and run the cell.

In [ ]:
# ── YOUR CUSTOM TICKET ─────────────────────────────────────────────────
MY_TICKET = "The auth service is returning 502 errors on prod-api-01"

state = await run_and_display(MY_TICKET, user_id="EMP-0999")

## 11. Memory Inspection
See what got saved to memory after the above runs.

In [ ]:
from memory.episodic_store import EpisodicStore
from memory.lt_store import LTStore

# ── Recent episodes ────────────────────────────────────────────────────
ep_store = EpisodicStore()
from infrastructure.db.supabase_client import get_supabase

sb = get_supabase()
result = sb.table("episodes").select("session_id, ticket_id, ticket_type, resolved, created_at").order("created_at", desc=True).limit(5).execute()

print("=" * 60)
print("RECENT EPISODES (Episodic Memory)")
print("=" * 60)
if result.data:
    for ep in result.data:
        print(f"  {ep['ticket_id']} | Type {ep['ticket_type']} | Resolved: {ep['resolved']} | {ep['created_at']}")
else:
    print("  No episodes yet — run a ticket through the swarm first")

In [ ]:
# ── Long-term facts ────────────────────────────────────────────────────
result = sb.table("lt_facts").select("fact, confidence, category, created_at").order("created_at", desc=True).limit(10).execute()

print("=" * 60)
print("LONG-TERM MEMORY FACTS")
print("=" * 60)
if result.data:
    for f in result.data:
        print(f"  [{f['category']:10s}] {f['confidence']:.0%}  {f['fact']}")
else:
    print("  No LT facts yet — facts are extracted after ticket resolution")

In [ ]:
# ── Ticket history ─────────────────────────────────────────────────────
result = sb.table("tickets").select("id, ticket_type, severity, status, created_at").order("created_at", desc=True).limit(10).execute()

type_labels = {1: "Access", 2: "Asset", 3: "Degradation", 4: "Outage"}

print("=" * 60)
print("TICKET HISTORY")
print("=" * 60)
if result.data:
    for t in result.data:
        label = type_labels.get(t.get('ticket_type'), '?')
        print(f"  {t['id'][:16]:20s} | {label:12s} | {t['severity']:8s} | {t['status']}")
else:
    print("  No tickets yet")

## 12. Qdrant Collection Stats

In [ ]:
from infrastructure.db.qdrant_client import get_qdrant

qdrant = get_qdrant()
collections = await qdrant.get_collections()

print("=" * 60)
print("QDRANT COLLECTIONS")
print("=" * 60)

for col in collections.collections:
    info = await qdrant.get_collection(col.name)
    print(f"  {col.name:25s} : {info.points_count:5d} points")